**CELL 27** Re-split cleaned data and cache



In [ ]:
from pyspark.storagelevel import StorageLevel

train_df, test_df = df_clean.randomSplit([0.8, 0.2], seed=42)

train_df = train_df.persist(StorageLevel.MEMORY_AND_DISK)
test_df  = test_df.persist(StorageLevel.MEMORY_AND_DISK)

print("Train:", train_df.count(), "Test:", test_df.count())

Train: 4788218 Test: 1197807


**CELL 28** Refit feature pipeline after numeric cleaning



In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

LABEL_COL = "SEVERITY"

label_indexer = StringIndexer(inputCol=LABEL_COL, outputCol="label", handleInvalid="keep")

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in cat_cols]

encoder = OneHotEncoder(
    inputCols=[f"{c}_idx" for c in cat_cols],
    outputCols=[f"{c}_ohe" for c in cat_cols],
    handleInvalid="keep"
)

assembler_inputs = [f"{c}_ohe" for c in cat_cols] + num_cols

assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="features_raw", handleInvalid="keep")

scaler = StandardScaler(inputCol="features_raw", outputCol="features", withStd=True, withMean=False)

feat_pipe = Pipeline(stages=[label_indexer] + indexers + [encoder, assembler, scaler]).fit(train_df)

train_ready = feat_pipe.transform(train_df).select("label", "features")
test_ready  = feat_pipe.transform(test_df).select("label", "features")

print("Prepared train rows:", train_ready.count(), "Prepared test rows:", test_ready.count())

Prepared train rows: 4788218 Prepared test rows: 1197807


**CELL 29** Verify no NaN/Inf inside the feature vector


In [ ]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F

# I verify that my feature vectors contain no NaN/Inf
arr_df = train_ready.select(vector_to_array("features").alias("fa"))

bad = arr_df.select(
    F.sum(
        F.expr("aggregate(fa, 0, (acc, x) -> acc + IF(isnan(x) OR x = double('inf') OR x = -double('inf'), 1, 0))")
    ).alias("bad_values")
).collect()[0]["bad_values"]

print("Bad values in train features:", bad)

Bad values in train features: 0


**11) Final Training Run + Final Comparison Table**

**CELL 30 R**etrain 4 models and produce final metrics table


In [ ]:
from pyspark.sql import Row
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier, LinearSVC, OneVsRest, MultilayerPerceptronClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.sql import functions as F

# Evaluators
eval_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
eval_f1  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
eval_wp  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
eval_wr  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")

num_classes = int(train_ready.select("label").distinct().count())
input_size = train_ready.select("features").first()["features"].size
layers = [input_size, 64, 32, num_classes]
print("Classes:", num_classes, " | MLP layers:", layers)

models = {}
models["Logistic Regression"] = LogisticRegression(featuresCol="features", labelCol="label", maxIter=50)
models["Decision Tree"] = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=10, minInstancesPerNode=50)

svm_binary = LinearSVC(featuresCol="features", labelCol="label", maxIter=50, regParam=0.1)
models["SVM (OneVsRest LinearSVC)"] = OneVsRest(classifier=svm_binary, labelCol="label", featuresCol="features")

models["Deep Learning (MLP)"] = MultilayerPerceptronClassifier(
    featuresCol="features", labelCol="label",
    layers=layers, maxIter=50, seed=42, blockSize=256
)

results = []

for name, model in models.items():
    print("\n==============================")
    print("Training:", name)
    print("==============================")

    fitted = model.fit(train_ready)
    preds  = fitted.transform(test_ready).cache()

    acc = float(eval_acc.evaluate(preds))
    f1  = float(eval_f1.evaluate(preds))
    wp  = float(eval_wp.evaluate(preds))
    wr  = float(eval_wr.evaluate(preds))

    print(f"{name} -> Accuracy: {acc:.4f} | F1: {f1:.4f} | W-Precision: {wp:.4f} | W-Recall: {wr:.4f}")

    results.append(Row(Model=name, Accuracy=acc, F1=f1, WeightedPrecision=wp, WeightedRecall=wr))

    preds.unpersist()

results_df = spark.createDataFrame(results).orderBy(F.desc("F1"))
results_df.show(truncate=False)

Classes: 4  | MLP layers: [197, 64, 32, 4]

Training: Logistic Regression
Logistic Regression -> Accuracy: 0.9929 | F1: 0.9926 | W-Precision: 0.9930 | W-Recall: 0.9929

Training: Decision Tree
Decision Tree -> Accuracy: 0.9932 | F1: 0.9930 | W-Precision: 0.9931 | W-Recall: 0.9932

Training: SVM (OneVsRest LinearSVC)
SVM (OneVsRest LinearSVC) -> Accuracy: 0.9167 | F1: 0.9131 | W-Precision: 0.9247 | W-Recall: 0.9167

Training: Deep Learning (MLP)
Deep Learning (MLP) -> Accuracy: 0.6463 | F1: 0.5074 | W-Precision: 0.4177 | W-Recall: 0.6463
+-------------------------+------------------+------------------+------------------+------------------+
|Model                    |Accuracy          |F1                |WeightedPrecision |WeightedRecall    |
+-------------------------+------------------+------------------+------------------+------------------+
|Decision Tree            |0.9931683484901992|0.9929643067073213|0.9930656850715233|0.9931683484901992|
|Logistic Regression      |0.992942101690

**12) Confusion Matrix for Best Model**

**CELL 31** Identify best model + confusion matrix counts



In [ ]:
best_model_name = results_df.first()["Model"]
print("Best model by F1:", best_model_name)

best_fitted = models[best_model_name].fit(train_ready)

best_preds = (
    best_fitted.transform(test_ready)
              .select("label", "prediction")
              .cache()
)

# Confusion matrix counts: label vs prediction
confusion_df = (
    best_preds.groupBy("label", "prediction")
              .count()
              .orderBy("label", "prediction")
)

confusion_df.show(200, truncate=False)

best_preds.unpersist()

Best model by F1: Decision Tree
+-----+----------+------+
|label|prediction|count |
+-----+----------+------+
|0.0  |0.0       |773312|
|0.0  |1.0       |606   |
|0.0  |3.0       |196   |
|1.0  |0.0       |3720  |
|1.0  |1.0       |349634|
|1.0  |3.0       |9     |
|2.0  |0.0       |187   |
|2.0  |2.0       |58919 |
|2.0  |3.0       |331   |
|3.0  |0.0       |2840  |
|3.0  |2.0       |294   |
|3.0  |3.0       |7759  |
+-----+----------+------+



DataFrame[label: double, prediction: double]

**13) Export Results for Report / Tableau**

**CELL 32** Export model comparison table + confusion matrix table to CSV



In [ ]:
import glob, shutil, os

# --- 1) Export model comparison results ---
OUT_RESULTS_DIR = "/content/model_comparison_results"

(
    results_df.coalesce(1)
              .write.mode("overwrite")
              .option("header", "true")
              .csv(OUT_RESULTS_DIR)
)

part_results = glob.glob(f"{OUT_RESULTS_DIR}/part-*.csv")[0]
FINAL_RESULTS_CSV = "/content/model_comparison_results.csv"
shutil.copy(part_results, FINAL_RESULTS_CSV)

print("Saved model comparison CSV:", FINAL_RESULTS_CSV)


# --- 2) Export confusion matrix
OUT_CONF_DIR = "/content/confusion_matrix_counts"

try:
    (
        confusion_df.coalesce(1)
                    .write.mode("overwrite")
                    .option("header", "true")
                    .csv(OUT_CONF_DIR)
    )

    part_conf = glob.glob(f"{OUT_CONF_DIR}/part-*.csv")[0]
    FINAL_CONF_CSV = "/content/confusion_matrix_counts.csv"
    shutil.copy(part_conf, FINAL_CONF_CSV)

    print("Saved confusion matrix CSV:", FINAL_CONF_CSV)

except NameError:
    print("confusion_df not found. Please run CELL 7 first if you want the confusion matrix export.")


# --- 3) Show files in /content so I can download them ---
!ls -lh /content | sed -n '1,200p'

Saved model comparison CSV: /content/model_comparison_results.csv
Saved confusion matrix CSV: /content/confusion_matrix_counts.csv
total 2.5G
drwxr-xr-x 2 root root 4.0K Feb 10 01:53 confusion_matrix_counts
-rw-r--r-- 1 root root  176 Feb 10 01:53 confusion_matrix_counts.csv
drwxr-xr-x 2 root root 4.0K Feb 10 01:53 model_comparison_results
-rw-r--r-- 1 root root  430 Feb 10 01:53 model_comparison_results.csv
-rw-r--r-- 1 root root 1.2G Feb  9 18:28 NYPD_Arrests_Data__Historic_.csv
drwxr-xr-x 2 root root 4.0K Feb  9 19:11 NYPD_Tableau_Cleaned
-rw-r--r-- 1 root root 1.3G Feb  9 19:12 NYPD_Tableau_Cleaned.csv
drwxr-xr-x 1 root root 4.0K Jan 16 14:24 sample_data


**CELL 33** Download model comparison CSV


In [ ]:
from google.colab import files
files.download("/content/model_comparison_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Create a Direct Colab Model Comparism CSV Table**

In [ ]:
import pandas as pd

data = {
    "Model": [
        "Decision Tree",
        "Logistic Regression",
        "SVM (OneVsRest LinearSVC)",
        "Deep Learning (MLP)"
    ],
    "Accuracy": [
        0.9931683484901992,
        0.99294210169084,
        0.9167152972056433,
        0.6462752346580042
    ],
    "F1": [
        0.9929643067073213,
        0.9925716602114856,
        0.9130844253402577,
        0.5074148109477526
    ],
    "WeightedPrecision": [
        0.9930656850715233,
        0.992965313352576,
        0.9246960137674678,
        0.4176725671786969
    ],
    "WeightedRecall": [
        0.9931683484901992,
        0.99294210169084,
        0.9167152972056434,
        0.6462752346580042
    ]
}

df = pd.DataFrame(data)
df.to_csv("model_results.csv", index=False)

print("CSV file created successfully!")

CSV file created successfully!


In [ ]:
from google.colab import files
files.download("model_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from pyspark import StorageLevel

train_df, test_df = df_feat.randomSplit([0.8, 0.2], seed=42)
train_df = train_df.persist(StorageLevel.MEMORY_AND_DISK)
test_df  = test_df.persist(StorageLevel.MEMORY_AND_DISK)

print("Train:", train_df.count(), "Test:", test_df.count())

Train: 4789063 Test: 1196962


In [ ]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator_f1  = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
evaluator_acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

lr = LogisticRegression(labelCol="label", featuresCol="features", maxIter=50)

lr_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.0, 0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)

lr_cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=lr_grid,
    evaluator=evaluator_f1,
    numFolds=3,
    parallelism=2,
    seed=42
)

lr_cv_model = lr_cv.fit(train_df)
best_lr = lr_cv_model.bestModel

print("LR best regParam:", best_lr.getRegParam())
print("LR best elasticNetParam:", best_lr.getElasticNetParam())
print("LR best CV F1:", float(max(lr_cv_model.avgMetrics)))

lr_test_preds = best_lr.transform(test_df)
print("LR test F1:", float(evaluator_f1.evaluate(lr_test_preds)))
print("LR test ACC:", float(evaluator_acc.evaluate(lr_test_preds)))

LR best regParam: 0.0
LR best elasticNetParam: 0.0
LR best CV F1: 0.9936149031284978
LR test F1: 0.9937391523874562
LR test ACC: 0.9939371508869955
